# Palace run registry

This notebook lives in the **registry folder** (e.g. `palace_runs/`) and is
how the table gets built and grown. Two ways runs get in:

* automatically from `palace_pipeline.ipynb` cell 8 after each run, or
* manually here — point cell 2 at any run folder (with its `.o` log
  and/or its `postpro_<tag>/` folder copied back from the HPC) and run it.

Cell 3 shows the table inline; the full report with per-pass mode tables
and design screenshots is `runlog.html` in this folder.

In [22]:
# ------------------------- settings -------------------------
PIPELINE_DIR = r"..\pipeline"          # folder containing palace_runlog.py
RUN_DIR = r"\\qcrew_drive\home\vashti_enjoy\cavity_pin_loss_chip_JJ_export_20260807_160052"   # run to record
LOG_FILE     = "palace_run3.o1291233"                     # e.g. "palace_eigen.o1290123", or None
POSTPRO      = "postpro_run3"                     # e.g. "postpro_run1" (inside RUN_DIR), or
                                        #   None to auto-detect from the config
SCREENSHOT   = "Screenshot2026-08-11 114448.png"                     # e.g. "design.png" (inside RUN_DIR), or None
NOTES = """mesh params:
JJ = 0.0002 mm
thin_lead = 0.0002 mm
medium_lead = 0.003 mm
pads = 0.2 mm
pin = 1 mm
Pin_1 = 0.4 mm
cavity = 2 mm"""

import os, sys, types
sys.path.insert(0, os.path.abspath(PIPELINE_DIR))
import palace_runlog as prl
REGISTRY = os.path.abspath(".")
print("registry:", REGISTRY)

registry: \\qcrew_drive\home\vashti_enjoy\palace_pipeline_v2\palace_pipeline


In [23]:
# ------------------------- add the run -------------------------
args = types.SimpleNamespace(
    log=os.path.join(RUN_DIR, LOG_FILE) if LOG_FILE else None,
    postpro=os.path.join(RUN_DIR, POSTPRO) if POSTPRO else None,
    run_dir=RUN_DIR,
    registry=REGISTRY,
    screenshot=os.path.join(RUN_DIR, SCREENSHOT) if SCREENSHOT else None,
    notes=NOTES,
)
prl.add_run(args)

recorded run 1291233: status=OOM_KILLED, 5 AMR its, 5 pass(es), final tets=698551
report: \\qcrew_drive\home\vashti_enjoy\palace_pipeline_v2\palace_pipeline\runlog.html


In [24]:
# ------------------------- view the table -------------------------
import csv
from IPython.display import HTML, display
with open(os.path.join(REGISTRY, "runs.csv"), encoding="utf-8") as fh:
    rows = list(csv.reader(fh))
table = "<table><tr>" + "".join(f"<th>{c}</th>" for c in rows[0]) + "</tr>"
for r in reversed(rows[1:]):
    table += "<tr>" + "".join(f"<td>{c}</td>" for c in r) + "</tr>"
table += "</table>"
display(HTML(table))
print("full report with per-pass detail + screenshots: runlog.html")

run_id,date,design,tag,status,amr_iterations,passes,tets_initial,tets_final,wall_clock,peak_mem_gb,f1_GHz,Q1,f2_GHz,Q2,f3_GHz,Q3,f4_GHz,Q4,f5_GHz,Q5,notes
1291233,2026-08-11 11:58,cavity_pin_loss_chip_JJ,run3,OOM_KILLED,5,5,389576,698551,1:44:16,255.9,4.082769,57169040.0,6.176765,502388.9,11.01079,5.975579,12.1694,32740.71,18.04404,221784.1,mesh params: JJ = 0.0002 mm thin_lead = 0.0002 mm medium_lead = 0.003 mm pads = 0.2 mm pin = 1 mm Pin_1 = 0.4 mm cavity = 2 mm


full report with per-pass detail + screenshots: runlog.html


In [18]:
import json, os

registry = REGISTRY  # or wherever your registry folder is
path = os.path.join(registry, "runs.jsonl")

with open(path, encoding="utf-8") as fh:
    records = [json.loads(line) for line in fh if line.strip()]

# find the run(s) you want gone — inspect first, don't guess
for r in records:
    print(r["run_id"], r["date"], r.get("design"), r.get("notes", "")[:40])

1291233 2026-08-11 11:51 cavity_pin_loss_chip_JJ what this run tests / which mesh sizes
1291233 2026-08-11 11:55 cavity_pin_loss_chip_JJ mesh params:
JJ = 0.0002 mm
thin_lead = 


In [19]:
run_id_to_remove = "1291233"   # whatever it printed above
records = [r for r in records if r["run_id"] != run_id_to_remove]

with open(path, "w", encoding="utf-8") as fh:
    for r in records:
        fh.write(json.dumps(r) + "\n")

In [20]:
import sys
sys.path.insert(0, os.path.abspath(PIPELINE_DIR))
import palace_runlog as prl
prl._write_csv(registry)
prl._write_html(registry)